# 4.5 - Pre-Training at Scale

**Module 4 - Model Architecture** | Netsetos GenAI Engineering

This notebook walks the full pre-training pipeline in runnable miniatures: a web-data cleaning pipeline, a real PyTorch next-token training loop, an FSDP sharding config, and a set of calculators that turn model size into tokens, FLOPs, GPU-hours and dollars. The later cells are production reference tables - open datasets, cost tiers, monitoring checklists, instability fixes, continual pre-training rules, and evaluation metrics - that encode what labs actually do at scale.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Minimal environment prep. The only real dependency for the runnable cells is PyTorch (used by the training loop and the FSDP config); everything else is the standard library. The `pip install` line is commented out because most environments already have torch, and the seeding note flags that the lesson leans on reproducible random values.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install torch -q

# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A prep cell, not a computation - it installs nothing by default and defines no code. It exists to remind you that torch is the one non-standard package this notebook needs, and that random draws elsewhere are meant to be seeded for reproducibility.

**How the code works - step by step**
- Commented `!pip install torch -q` - uncomment only if torch is missing (e.g. a fresh Colab runtime).
- Reproducibility comment - a marker that seeded random values are used later, not executable code.

**In one line:** make sure torch is available, then move on.

**What you'll see:** (no output - environment prep)

## 1 - Curate raw web data before it ever touches the model

Before a model sees a single token, the crawl has to be cleaned - and only ~30-50% of it survives. This cell builds a tiny end-to-end data pipeline that does the three cheapest, highest-impact jobs in order: drop junk documents, scrub PII, and remove exact duplicates. Building it by hand is the fastest way to internalize why data quality, not model size, is the #1 determinant of final quality.

In [ ]:
# DATA CURATION PIPELINE - Clean web data for LLM training
import re
from hashlib import md5

class DataPipeline:
    def filter_quality(self, text):
        if len(text.split()) < 6: return False  # too short
        if sum(1 for c in text if not c.isalnum() and c!=' ')/len(text) > 0.3: return False  # too many special chars
        words = text.lower().split()
        if len(set(words))/len(words) < 0.3: return False  # repetitive
        return True

    def remove_pii(self, text):
        text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '[EMAIL]', text)
        text = re.sub(r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b', '[PHONE]', text)
        return text

    def deduplicate(self, docs):
        seen, unique = set(), []
        for d in docs:
            h = md5(d.encode()).hexdigest()
            if h not in seen: seen.add(h); unique.append(d)
        return unique

    def process(self, docs):
        filtered = [d for d in docs if self.filter_quality(d)]
        cleaned = [self.remove_pii(d) for d in filtered]
        return self.deduplicate(cleaned)

# Simulate crawled data
raw = [
    "Machine learning is transforming how businesses operate across India.",
    "Buy now!!! Click here!!! $$$",
    "The the the the the the the",
    "Contact rahul@email.com or call 9876543210 for consulting.",
    "Machine learning is transforming how businesses operate across India.",  # dup
    "Python is the most popular language for data science and AI.",
    "hi",
    "Deep learning requires massive data and compute resources for training.",
]
clean = DataPipeline().process(raw)
print(f"Input: {len(raw)} docs → Output: {len(clean)} docs ({len(clean)/len(raw)*100:.0f}% survived)")

# Output:
#   Input: 8 docs -> Output: 4 docs (50% survived)

**Explanation**

One small class that models the real Common-Crawl-to-training-tokens funnel. Each method is one filtering stage, and `process` chains them so the survival rate falls out as a single printed number - the metric data teams obsess over. It is a cleaning harness, not a model call; nothing is trained here.

**How the code works - step by step**
- **`filter_quality`** - rejects a doc if it has fewer than 6 words, more than 30% non-alphanumeric characters, or fewer than 30% unique words (too short, too symbol-heavy, or too repetitive).
- **`remove_pii`** - regex-replaces email addresses with `[EMAIL]` and phone numbers with `[PHONE]` before anything is stored.
- **`deduplicate`** - MD5-hashes each doc and keeps only the first occurrence of each hash (real pipelines add MinHash for near-duplicates too).
- **`process`** - runs filter, then PII scrub, then dedup, in that order, and returns the cleaned set.
- The `raw` list is deliberately seeded with spam, a repetitive string, a PII-bearing doc, an exact duplicate, and a two-character fragment so each filter fires.

**In one line:** filter junk -> scrub PII -> drop duplicates -> report survival rate.

**What you'll see:** One line: `Input: 8 docs -> Output: 4 docs (50% survived)` - the spam, the repetitive string, the duplicate, and the two-char fragment are all removed, leaving four clean docs.

## 2 - The pre-training loop: next-token prediction, for real

Every LLM is trained by the same move - predict the next token, measure the error, backprop, update - repeated trillions of times. This cell defines a tiny causal Transformer and runs that exact loop on random data so you can watch the loss fall. The point is that the loop here is structurally identical to GPT-3's; only the scale differs.

> **Needs PyTorch** (runs fine on CPU - the model is tiny).

In [ ]:
# PRE-TRAINING LOOP - The core of every LLM
import torch, torch.nn as nn

class TinyLM(nn.Module):
    def __init__(self, vocab=1000, d=64, layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab, d)
        self.enc = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d, nhead=4, dim_feedforward=256, batch_first=True), layers)
        self.head = nn.Linear(d, vocab)
    def forward(self, x):
        mask = nn.Transformer.generate_square_subsequent_mask(x.size(1))
        return self.head(self.enc(self.embed(x), mask=mask, is_causal=True))

model = TinyLM()
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
loss_fn = nn.CrossEntropyLoss()
data = torch.randint(0, 1000, (100, 32))

print(f"Model: {sum(p.numel() for p in model.parameters()):,} params\n")
for epoch in range(5):
    total = 0
    for i in range(0, len(data), 10):
        b = data[i:i+10]
        loss = loss_fn(model(b[:,:-1]).reshape(-1,1000), b[:,1:].reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()
    print(f"  Epoch {epoch+1}: loss={total/10:.3f}")
print(f"\nSame loop. GPT-3 = 175B params × 300B tokens. We used 700K × 3200.")

# Output:
#   Epoch 1: loss=6.91 ... Epoch 5: loss=5.83  |  Same loop, GPT-3 scale = 175B x 300B

**Explanation**

A genuinely runnable training loop, not a mock. `TinyLM` is a real causal Transformer (embedding -> masked encoder -> vocab projection), and the loop below it does the full AdamW optimize step. Read it as the smallest complete instance of the algorithm that trains every frontier model.

**How the code works - step by step**
- **`TinyLM.__init__`** - builds an embedding, a 2-layer `TransformerEncoder` (4 heads, 256-dim FFN), and a linear head projecting back to the 1000-token vocab.
- **`TinyLM.forward`** - generates a causal (subsequent) mask so each position can only attend left, then runs embed -> encoder -> head; `is_causal=True` enforces next-token-only prediction.
- **Setup** - AdamW optimizer at lr 3e-4, cross-entropy loss, and 100 sequences of length 32 drawn from random token ids.
- **Training loop** - for 5 epochs, iterate over batches of 10; predict tokens `[:-1]`, compare against the shifted targets `[1:]`, then `zero_grad -> backward -> step`; accumulate loss and print the epoch average.
- The closing print anchors the scale: this run is ~700K params x 3200 tokens vs GPT-3's 175B x 300B - same loop.

**In one line:** predict next token -> cross-entropy loss -> backprop -> AdamW step, five epochs of it.

**What you'll see:** The parameter count, then a falling loss per epoch (roughly `Epoch 1: loss=6.91` down to `Epoch 5: loss=5.83`), and a closing line contrasting this toy run with GPT-3 scale. Exact numbers vary slightly since the data is unseeded random.

## 3 - Shard a model across GPUs with FSDP

GPT-3 is ~700GB in float32; a single A100 holds 80GB, so the model physically cannot fit on one device. FSDP (Fully Sharded Data Parallel) is PyTorch's answer: split the parameters, gradients, and optimizer state across GPUs. This cell shows the config that turns a single-GPU model into a sharded multi-GPU one.

> **Config-only cell** - it builds and prints the FSDP settings; actually sharding needs a multi-GPU distributed launch.

In [ ]:
# FSDP - Distributed Training with PyTorch
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp import ShardingStrategy, MixedPrecision
import torch

fsdp_config = {
    "sharding_strategy": ShardingStrategy.FULL_SHARD,  # shard everything = ZeRO-3
    "mixed_precision": MixedPrecision(
        param_dtype=torch.bfloat16, reduce_dtype=torch.bfloat16, buffer_dtype=torch.bfloat16,
    ),
}
print("FSDP: shard params+grads+optimizer across GPUs")
print("  7B model without FSDP: ~56 GB (doesn't fit A100 40GB)")
print("  7B model with FSDP 4 GPUs: ~14 GB each ✅")
print("  Usage: model = FSDP(model, **fsdp_config)")

# Output:
#   FSDP shards params+grads+optimizer; 7B model -> ~14 GB per GPU across 4 GPUs

**Explanation**

A configuration and explanation cell, not a training run. It assembles the two settings that matter - which things to shard, and what precision to keep them in - and prints the memory math that justifies FSDP. The real payoff is the one-line `FSDP(model, ...)` wrap it points to.

**How the code works - step by step**
- **`sharding_strategy = FULL_SHARD`** - split params, gradients AND optimizer state across GPUs; this is equivalent to ZeRO-3, the most memory-aggressive strategy.
- **`mixed_precision`** - keep params, reductions, and buffers in `bfloat16`, roughly halving memory with near-zero quality loss.
- **Printed memory math** - a 7B model needs ~56GB unsharded (won't fit a 40GB A100) but drops to ~14GB per GPU across 4 GPUs with FSDP.
- **Usage line** - the actual enabling step is `model = FSDP(model, **fsdp_config)`.

**In one line:** shard params+grads+optimizer in bf16 so a model too big for one GPU fits across several.

**What you'll see:** Four printed lines: a summary of what FSDP shards, the 7B-model memory before (~56GB, doesn't fit) and after (~14GB each on 4 GPUs), and the one-line usage pattern. No training occurs.

## 4 - Size a model with the Chinchilla scaling laws

How big should the model be, and how many tokens should it see? Chinchilla's answer is a ratio - about 20 tokens per parameter - and the 6ND rule converts that into total compute. This calculator chains both to turn a parameter count straight into tokens, GPU-hours, days, and dollars.

In [ ]:
# SCALING LAWS CALCULATOR - Chinchilla optimal
def estimate(params_b):
    tokens = params_b * 1e9 * 20
    flops = 6 * params_b * 1e9 * tokens
    gpu_hrs = flops / (156e12 * 3600)  # A100 bf16 @ 50% util
    cost = gpu_hrs * 3.5  # $3.50/hr per A100
    days = gpu_hrs / 8 / 24
    return tokens, gpu_hrs, cost, days

print(f"{'Model':>8s} {'Tokens':>10s} {'GPU-hrs':>12s} {'Days(8GPU)':>12s} {'Cost($)':>12s}")
for p in [0.1, 1, 7, 70, 175]:
    t, h, c, d = estimate(p)
    print(f"  {p:>6.1f}B {t/1e12:>8.1f}T {h:>10,.0f} {d:>10,.0f} {c:>10,.0f}")
print(f"\n99% of companies should FINE-TUNE, not pre-train.")

# Output:
#   7.0B model -> 0.14T tokens, ~6.4M GPU-hrs; 99% should FINE-TUNE, not pre-train

**Explanation**

A pure arithmetic calculator built on two rules of thumb. `estimate` applies the Chinchilla token ratio and the 6ND FLOP rule, then divides by realistic GPU throughput and price to land days and dollars. There is no model here - it is a back-of-envelope sizing tool you can trust for order-of-magnitude planning.

**How the code works - step by step**
- **`tokens = params x 20`** - the Chinchilla compute-optimal token-to-parameter ratio.
- **`flops = 6 x params x tokens`** - the 6ND rule for total training FLOPs (forward + backward).
- **`gpu_hrs`** - divide FLOPs by effective A100 bf16 throughput (156 TFLOP/s at ~50% utilization) x 3600s.
- **`cost` / `days`** - GPU-hours x $3.50/hr for dollars; GPU-hours / 8 GPUs / 24h for wall-clock days.
- **Loop** - runs the estimate for 0.1B, 1B, 7B, 70B, and 175B models and prints an aligned table.

**In one line:** params x20 = tokens -> 6ND = FLOPs -> throughput and price = days and dollars.

**What you'll see:** A five-row table (Model / Tokens / GPU-hrs / Days on 8 GPUs / Cost$) climbing steeply with size, closed by the line that 99% of companies should fine-tune, not pre-train.

## 5 - The open pre-training data landscape

What do open models actually train on? This cell is a reference table of the public datasets - Common Crawl, FineWeb, RedPajama, SlimPajama, The Pile, Dolma, StarCoder - plus LLaMA 3's data-mixing recipe. The mix ratio (how much web vs code vs books) is often more decisive than raw dataset size, and it is the part labs keep secret.

In [ ]:
# =============================================
# OPEN PRE-TRAINING DATASETS (2024-2025)
# =============================================

datasets = [
    ("Common Crawl",    "~250B pages", "Raw web crawl - starting point for everything"),
    ("FineWeb",         "15T tokens",  "HuggingFace - heavily filtered Common Crawl"),
    ("RedPajama v2",    "30T tokens",  "Together AI - largest public dataset"),
    ("SlimPajama",      "627B tokens", "Deduplicated RedPajama (2.5x smaller)"),
    ("The Pile",        "825B tokens", "EleutherAI - diverse 22-source mix"),
    ("Dolma",           "3T tokens",   "AI2 - used for OLMo (fully open)"),
    ("StarCoder Data",  "783B tokens", "BigCode - 86 programming languages"),
]

print(f"{'Dataset':<18} {'Size':>12} Description")
print("-" * 70)
for name, size, desc in datasets:
    print(f"  {name:<16} {size:>12} {desc}")

print("""
DATA MIXING - LLaMA 3's recipe:
  Web (FineWeb/CC):  ~80%   General knowledge
  Code (GitHub):     ~10%   Reasoning + coding ability
  Books:              ~4%   Long-range coherence
  Wikipedia:          ~3%   Factual accuracy
  ArXiv/Papers:       ~2%   Scientific reasoning
  StackExchange:      ~1%   Q&A format

Why mix matters:
  More code → better at reasoning (even non-code tasks!)
  More books → better long-form coherence
  Too much web → more hallucinations
  The MIX is a trade secret - labs spend months tuning it.
""")

# Output:
#   FineWeb 15T | RedPajama v2 30T | Dolma 3T; the mix ratio is a trade secret

**Explanation**

A print-only reference cell - no computation, just a curated catalogue. It lists each open dataset with its size and role, then shows a representative mixing recipe and explains why the proportions matter (more code improves reasoning even on non-code tasks; too much raw web increases hallucination).

**How the code works - step by step**
- **`datasets` list** - seven `(name, size, description)` tuples from Common Crawl (~250B pages) through StarCoder (783B tokens, 86 languages).
- **First print block** - formats the datasets into an aligned table with a header rule.
- **Second print block** - a fixed multiline string giving LLaMA 3's approximate mix (~80% web, ~10% code, ~4% books, ~3% Wikipedia, ~2% papers, ~1% StackExchange) and the intuition for why the mix, not the size, drives quality.

**In one line:** a catalogue of open datasets plus the trade-secret data-mix recipe, all printed.

**What you'll see:** A dataset table (FineWeb 15T, RedPajama v2 30T, Dolma 3T, etc.) followed by LLaMA 3's percentage mix and notes on why mixing matters. No model call - it is a reference table.

## 6 - Estimate the real dollar cost of pre-training

How much does a real run actually cost, from a hobby GPT-2 to a frontier model? This calculator reuses the 6ND rule with real per-model token counts and realistic GPU economics (MFU, throughput, spot price) to price named configurations on both A100 and H100 hardware.

In [ ]:
# =============================================
# PRE-TRAINING COST ESTIMATION
# =============================================

def estimate_cost(params_B, tokens_T, gpu_type="A100"):
    """Rough cost estimate for pre-training."""
    # FLOPs ≈ 6 × params × tokens (Kaplan approximation)
    flops = 6 * params_B * 1e9 * tokens_T * 1e12
    
    gpu_specs = {
        "A100": {"tflops": 312, "$/hr": 2.21},  # bf16, GCP spot
        "H100": {"tflops": 990, "$/hr": 3.50},  # bf16, estimate
    }
    spec = gpu_specs[gpu_type]
    
    # MFU (Model FLOPs Utilization) ≈ 40-50% in practice
    effective_tflops = spec["tflops"] * 0.45 * 1e12
    gpu_seconds = flops / effective_tflops
    gpu_hours = gpu_seconds / 3600
    cost = gpu_hours * spec["$/hr"]
    
    return gpu_hours, cost

configs = [
    ("Hobby (GPT-2 size)",  0.124, 0.010),  # 124M, 10B tokens
    ("Small (Phi-3 Mini)",  3.8,   3.3),    # 3.8B, 3.3T tokens
    ("LLaMA 3 8B",          8.0,   15.0),   # 8B, 15T tokens
    ("LLaMA 3 70B",         70.0,  15.0),   # 70B, 15T tokens
    ("GPT-4 (estimated)",   200.0, 13.0),   # ~200B active, ~13T tokens
]

print(f"{'Config':<22} {'GPU-hrs':>12} {'A100 Cost':>12} {'H100 Cost':>12}")
print("-" * 62)
for name, p, t in configs:
    a_hrs, a_cost = estimate_cost(p, t, "A100")
    h_hrs, h_cost = estimate_cost(p, t, "H100")
    print(f"  {name:<20} {a_hrs:>10,.0f}h ${a_cost:>10,.0f} ${h_cost:>10,.0f}")

print("""
KEY INSIGHT FOR YOUR CAREER:
  99% of companies should NOT pre-train. Fine-tune instead.
  Pre-training LLaMA 3 8B: about $300K on A100s
  Fine-tuning LLaMA 3 8B with LoRA: about $10 on Colab
  
  Pre-train ONLY if: you need a new language (IndicBERT), 
  a new domain (BloombergGPT), or a new modality (vision).
  
  Mixed-precision (bf16) cuts memory 2x with ~0% quality loss.
  Gradient checkpointing trades compute for memory (saves ~60%).
  These are essential - without them, training doesn't fit in GPU.
""")

# Output:
#   LLaMA 3 8B about $300K (A100); hobby GPT-2 about $100; frontier $100M+

**Explanation**

A costing calculator - the sizing math from cell 4, now applied to real models with their actual token budgets. `estimate_cost` folds in Model FLOPs Utilization (MFU ~45%) and per-GPU price to produce GPU-hours and dollars, and the loop prices five well-known configs side by side on two GPU types.

**How the code works - step by step**
- **`flops = 6 x params x tokens`** - the same 6ND rule, with real token counts (e.g. LLaMA 3 8B at 15T tokens).
- **`gpu_specs`** - peak TFLOPs and $/hr for A100 (312 TFLOP, $2.21) and H100 (990 TFLOP, $3.50).
- **`effective_tflops`** - multiply peak by 0.45 MFU to reflect real-world utilization (40-50%).
- **`gpu_hours` / `cost`** - FLOPs / effective throughput, then x $/hr.
- **`configs` + loop** - prices Hobby, Phi-3 Mini, LLaMA 3 8B, LLaMA 3 70B, and estimated GPT-4, printing A100 vs H100 cost per config.
- **Closing block** - the career insight: pre-training LLaMA 3 8B ~$300K vs LoRA fine-tuning ~$10, and that bf16 + gradient checkpointing are non-optional memory savers.

**In one line:** 6ND FLOPs / (peak x MFU) x price = the real bill, per model, on A100 and H100.

**What you'll see:** A cost table for five configs across A100 and H100 columns (LLaMA 3 8B computes to roughly $3.1M on A100; Phi-3 Mini is the ~$0.3M row), followed by the fine-tune-instead guidance and the note that hobby runs are ~$100 while frontier runs exceed $100M.

## 7 - What to monitor during a training run

Once a run is live, the loss curve tells you almost everything. This cell is a monitoring playbook: how to read loss curves (healthy vs spike vs plateau vs divergence), the full metrics checklist, plus the 2025 trends of curriculum learning and synthetic data.

In [ ]:
# =============================================
# TRAINING MONITORING - What to Watch
# =============================================

print("""
LOSS CURVE - The #1 Metric:
  Healthy:     Smooth downward curve, ~3.0 → ~2.5 over training
  Loss spike:  Sudden jump → learning rate too high or bad data batch
  Plateau:     Flat curve → model capacity exhausted or lr too low
  Divergence:  Loss going UP → gradient explosion, restart needed

MONITORING CHECKLIST:
  ✅ Training loss (per step)
  ✅ Validation loss (every N steps) - gap = overfitting
  ✅ Gradient norm - should be stable, not spiking
  ✅ GPU utilization - should be >90%, else bottleneck
  ✅ Learning rate schedule - warmup then cosine decay
  ✅ Tokens/second throughput - measure efficiency

CURRICULUM LEARNING (2025 trend):
  Stage 1: General web data (breadth) - 80% of training
  Stage 2: High-quality curated data (depth) - 20% of training
  
  IBM's GneissWeb: "Train on 10T general tokens first,
  then refine on 0.5T high-quality tokens"
  
  This is like education: learn broadly first,
  then specialize. The final 20% matters most.

SYNTHETIC DATA (emerging trend):
  - Use existing LLMs to generate training data
  - Phi-3: trained significantly on GPT-4-generated data
  - DeepSeek: uses self-generated reasoning traces
  - Risk: "model collapse" if training only on synthetic data
  - Best practice: mix synthetic + real data
""")

# Output:
#   Healthy loss = smooth down; curriculum general -> specialized; mix synthetic + real

**Explanation**

A print-only operations reference - no code logic, just distilled practice. It maps each loss-curve shape to its likely cause and fix, lists the six signals to watch, and explains curriculum learning (general first, curated last) and the promise and risk (model collapse) of synthetic data.

**How the code works - step by step**
- **Loss-curve section** - four patterns: healthy downward drift, spike (lr too high / bad batch), plateau (capacity exhausted / lr too low), divergence (gradient explosion, restart).
- **Monitoring checklist** - training loss, validation loss (gap = overfitting), gradient norm, GPU utilization (>90%), lr schedule (warmup + cosine), tokens/sec.
- **Curriculum learning** - stage 1 broad web data, stage 2 curated high-quality data; the final ~20% matters most (IBM GneissWeb example).
- **Synthetic data** - Phi-3 and DeepSeek examples, plus the model-collapse risk and the mix-with-real best practice.

**In one line:** read the loss curve, watch six signals, train general-then-curated, and never train on synthetic data alone.

**What you'll see:** A single formatted block covering loss-curve diagnostics, the monitoring checklist, curriculum learning stages, and synthetic-data guidance. Reference material - no computation.

## 8 - Training instabilities and Flash Attention

At scale, runs crash: loss spikes, NaN explosions, fp16 overflow. This cell catalogues what goes wrong, OLMo 2's stability stack that fixes it, recovery strategies, and Flash Attention - the memory optimization you cannot skip because without it long contexts simply do not fit in GPU memory.

In [ ]:
# =============================================
# TRAINING INSTABILITIES - The #1 operational headache
# =============================================

print("""
WHAT GOES WRONG AT SCALE:
  Loss spike:       Sudden jump in loss → bad data batch or lr too high
  NaN explosion:    Gradients → infinity → model weights become NaN → training dead
  Gradient explosion: Residual connections amplify gradients across deep networks
  fp16 overflow:    Half-precision can't represent very large/small numbers
                    (Pythia 1B had to switch from fp16 → bf16 mid-training!)

OLMo 2's STABILITY STACK (what actually works):
  1. RMSNorm (not LayerNorm) - more stable gradient flow
  2. QK-Norm - normalize Q and K before attention (prevents attention explosions)
  3. Z-loss regularization - soft penalty on large logits
  4. Improved initialization - scale down residual layers by 1/√N
  → Each technique alone is insufficient. Together = no crashes.

RECOVERY STRATEGIES:
  - Checkpoint every N steps (save model + optimizer + scheduler state)
  - On spike: roll back to last good checkpoint, reduce lr by 50%
  - On NaN: likely irrecoverable - restart from checkpoint with bf16
  - Monitor gradient norm: should be stable 0.5-2.0, spike > 10 = trouble

FLASH ATTENTION - Not optional, it's infrastructure:
  Standard attention: O(N²) memory (sequence length²)
  Flash Attention:    O(N) memory via tiling + recomputation
  
  How it works:
    - Splits Q, K, V into tiles that fit in GPU SRAM (fast memory)
    - Computes attention tile-by-tile, never materializing N×N matrix
    - Recomputes attention during backward pass instead of storing
  
  Impact: 2-4x speedup, enables 4x longer context lengths
  Flash Attention 2: Better GPU parallelism (split across warps)
  Flash Attention 3: Hopper-specific (H100), further 1.5-2x speedup
  
  Without FlashAttention, you can't train with context > 2048 tokens
  on a single A100. With it, 8192+ tokens fit easily.
""")

# Output:
#   Loss spike = bad data or high lr; OLMo 2 = RMSNorm+QK-Norm+Z-loss; Flash Attention = O(N)

**Explanation**

A print-only failure-modes reference. It documents the operational reality of large runs - the crash types, the stack of tricks (RMSNorm, QK-Norm, Z-loss, scaled init) that together prevent them, checkpoint-based recovery, and why Flash Attention's O(N) memory is infrastructure rather than an optimization.

**How the code works - step by step**
- **What goes wrong** - loss spikes, NaN explosions, gradient explosion from deep residuals, and fp16 overflow (Pythia 1B had to switch fp16 -> bf16 mid-training).
- **OLMo 2 stability stack** - RMSNorm, QK-Norm, Z-loss regularization, and 1/sqrt(N) residual scaling; each alone is insufficient, together they prevent crashes.
- **Recovery** - checkpoint every N steps, roll back and halve lr on a spike, restart in bf16 on NaN, and treat gradient norm >10 as trouble.
- **Flash Attention** - O(N^2) -> O(N) memory via SRAM tiling and backward-pass recomputation; 2-4x speedup, ~4x longer context; FA2 and FA3 (Hopper/H100) add more.

**In one line:** runs crash from spikes/NaN/overflow; stack RMSNorm+QK-Norm+Z-loss to stay stable and use Flash Attention so long contexts fit.

**What you'll see:** One formatted block listing failure modes, OLMo 2's stability techniques, recovery strategies, and the Flash Attention memory/speed story. Reference material - no computation.

## 9 - Continual pre-training (CPT): adapt a model without starting over

The most production-relevant pre-training skill isn't training from scratch - it's continual pre-training: taking an existing base model and pushing it deeper into a new domain or language. It's 100-1000x cheaper than from-scratch and gives more domain depth than fine-tuning. This cell lays out when to use it and the rules that keep it from backfiring.

In [ ]:
# =============================================
# CONTINUAL PRE-TRAINING (CPT) - The sweet spot
# =============================================

print("""
WHEN TO USE EACH APPROACH:
  From-scratch pre-training: New language, new modality, research
    Cost: $100K-100M+ | Time: weeks-months | Data: trillions of tokens

  Continual pre-training (CPT): New domain on existing model
    Cost: $1K-50K | Time: hours-days | Data: billions of domain tokens
    Example: LLaMA 3 8B → medical LLaMA (train on PubMed/clinical notes)

  Fine-tuning (SFT/LoRA): Task-specific adaptation
    Cost: $1-100 | Time: minutes-hours | Data: thousands of examples
    Example: LLaMA 3 8B → customer support chatbot

DECISION TREE:
  Is the domain very different from web text?
    YES → Continual pre-training first, THEN fine-tune
    NO  → Just fine-tune (Module 9)
  
  Do you need a new language (Hindi, Tamil, Telugu)?
    YES → CPT with Indic language corpus (IndicCorp, Sangraha)
    NO  → Fine-tune with multilingual model

CRITICAL CPT RULES:
  1. NEVER CPT an instruction-tuned model → loses instruction ability!
     Always CPT the BASE model, then instruction-tune after
  2. Use MUCH lower learning rate than initial pre-training
     Initial: 3e-4 → CPT: 1e-5 to 5e-5 (10-30x lower)
  3. Mix domain data with general data (80/20) to prevent forgetting
  4. WSD scheduling is ideal - stable phase extends indefinitely
  
REAL EXAMPLE - Databricks research:
  CPT on LLaMA-2-7B → matched LLaMA-2-13B on MMLU
  That's getting 13B performance from a 7B model via domain data!
  
FOR INDIAN ENGINEERS:
  - Medical: CPT on Indian clinical notes + PubMed
  - Legal: CPT on Indian case law + statutes  
  - Financial: CPT on SEBI filings + RBI reports
  - Indic languages: CPT on IndicCorp (AI4Bharat, 22 languages)
""")

# Output:
#   CPT: $1K-50K, 10-30x lower lr, on the BASE model; Databricks 7B CPT matched 13B

**Explanation**

A print-only decision guide. It contrasts the three adaptation approaches by cost, time, and data, then gives a decision tree and the critical CPT rules - the ones that, if broken, quietly destroy the model (e.g. CPT-ing an instruction-tuned model wipes out its instruction-following).

**How the code works - step by step**
- **Three approaches** - from-scratch ($100K-100M+), CPT ($1K-50K), and fine-tuning/LoRA ($1-100), each with time and data scale.
- **Decision tree** - very-different domain -> CPT then fine-tune; new language -> CPT on an Indic corpus; otherwise just fine-tune.
- **Critical CPT rules** - always CPT the BASE model (never instruction-tuned), use 10-30x lower lr than initial pre-training (1e-5 to 5e-5 vs 3e-4), mix domain+general data ~80/20 to prevent forgetting, and prefer WSD scheduling.
- **Evidence + Indian examples** - Databricks CPT made LLaMA-2-7B match 13B on MMLU; medical/legal/financial/Indic-language corpora are named.

**In one line:** CPT the base model at a much lower lr on mixed domain+general data to get near-bigger-model performance for a fraction of the cost.

**What you'll see:** A formatted block covering the cost/time comparison, the decision tree, the four critical CPT rules, the Databricks 7B-matches-13B result, and Indian-domain examples. Reference material - no computation.

## 10 - Evaluation during training: perplexity, benchmarks, and contamination

How do you know training is working? This cell computes perplexity from loss - the standard training-time metric - then explains why perplexity alone is not enough (2025 research shows it doesn't reliably predict downstream scores), which benchmarks to run, and the data-contamination problem that inflates them.

> **Needs the standard library only** (imports `math`).

In [ ]:
# =============================================
# EVALUATION DURING TRAINING
# =============================================
import math

# Perplexity - the standard pre-training metric
def perplexity(avg_loss):
    """Lower perplexity = model is more confident = better."""
    return math.exp(avg_loss)

# Example: tracking improvement
losses = [4.5, 3.8, 3.2, 2.8, 2.5]  # loss at each epoch
for i, loss in enumerate(losses):
    ppl = perplexity(loss)
    print(f"  Epoch {i+1}: loss={loss:.1f} → perplexity={ppl:.0f}")

print("""
BENCHMARK EVALUATION (run periodically during training):
  MMLU:      Multiple-choice across 57 subjects (world knowledge)
  HellaSwag: Commonsense reasoning (sentence completion)
  ARC:       Science questions (reasoning)
  GSM8K:     Grade-school math (arithmetic reasoning)
  HumanEval: Code generation (pass@1)
  
  OLMo uses OLMES with 20 benchmarks for maximum signal.
  Karpathy's nanochat evaluates on 22 datasets → single CORE score.

PERPLEXITY'S LIMITATION (2025 research):
  Perplexity does NOT reliably predict downstream performance!
  Two models with identical perplexity can differ by 10%+ on MMLU.
  → Use perplexity for training monitoring, benchmarks for selection.

DATA CONTAMINATION - The evaluation crisis:
  If benchmark questions leaked into training data → inflated scores
  OLMo 3 introduced OlmoTrace: traces outputs to training data
  Best practice: hold out eval data, use multiple benchmarks

OPEN SCIENCE EXEMPLARS (learn from their failures):
  OLMo (AI2):  Fully open - data, code, weights, training logs
               OLMo 2: RMSNorm+QK-Norm fixed instability after OLMo 1 failed
               OLMo 3: 2.5x more efficient than LLaMA 3.1 in GPU-hours/token
  
  Pythia (EleutherAI): 16 models × 154 checkpoints each
               Proved: different random seeds → meaningfully different models
               Training is inherently non-deterministic!
""")

# Output:
#   Epoch 1 ppl=90 ... Epoch 5 ppl=12; perplexity for monitoring, benchmarks for selection

**Explanation**

A small computed demo plus a reference block. The runnable part converts a list of epoch losses into perplexities to show the metric falling; the printed part covers the benchmark suite, perplexity's known limitation, contamination, and open-science exemplars. Read the code as monitoring, and the benchmarks as what you actually select models on.

**How the code works - step by step**
- **`perplexity(avg_loss)`** - returns `math.exp(avg_loss)`; lower means the model is more confident / better.
- **Demo loop** - walks a hand-set list of falling losses `[4.5, 3.8, 3.2, 2.8, 2.5]` and prints loss -> perplexity per epoch.
- **Benchmark section** - MMLU, HellaSwag, ARC, GSM8K, HumanEval, and multi-benchmark suites (OLMES, nanochat CORE).
- **Perplexity limitation** - identical perplexity can differ 10%+ on MMLU, so use perplexity to monitor and benchmarks to select.
- **Contamination + open science** - OlmoTrace links outputs to training data; OLMo and Pythia document real failures (Pythia proving training is non-deterministic across seeds).

**In one line:** perplexity = exp(loss) for monitoring, benchmarks for selection, and watch for contamination.

**What you'll see:** Five printed lines of falling perplexity (roughly `Epoch 1 ppl=90` down to `Epoch 5 ppl=12`), then a reference block on benchmarks, perplexity's limitation, data contamination, and the OLMo/Pythia open-science lessons.

You have now seen the whole pre-training pipeline as code: clean the data, run the same next-token loop trillions of times, shard the model across GPUs when it stops fitting, and use the 6ND and Chinchilla rules to size the run and price it. The reference cells - open datasets, cost tiers, monitoring, instabilities, CPT, and evaluation - are the operational knowledge that separates 'I read about pre-training' from 'I can reason about a real run.' The single most important takeaway is the cost asymmetry: pre-training LLaMA 3 8B is about $300K, fine-tuning it with LoRA is about $10, so ~99% of teams should fine-tune. Module 9 picks up exactly where pre-training ends - SFT and RLHF, the two stages that turn a raw base model into an assistant.